# 0. Single Sample Debug Notebook

This notebook restores selected split artifacts, resolves one sample by `SENTENCE_NAME`, reviews source/gate/tier evidence, recomputes debug comparisons through the public workflow API, writes debug reports, and prints a final verdict.

This notebook is a thin operator console. It does not repair anything, mutate production artifacts, or read manifests/payloads directly.


# 1. Runtime Setup


## 1.1 Repository and roots


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/0xmillennium/text-to-sign-production.git"
REPO_REF = "chore/core-layout-notebook-workflows"
PROJECT_ROOT = Path("/content/text-to-sign-production")
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/text-to-sign-production")
RUNTIME_ROOT = PROJECT_ROOT / "runtime" / "debug"

print(f"Repository URL: {REPO_URL}")
print(f"Repository ref: {REPO_REF}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Drive project root: {DRIVE_PROJECT_ROOT}")
print(f"Runtime root: {RUNTIME_ROOT}")


## 1.2 Mount Drive


In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
if not DRIVE_PROJECT_ROOT.parent.is_dir():
    raise FileNotFoundError(f"Drive MyDrive root is missing: {DRIVE_PROJECT_ROOT.parent}")
print(f"Drive mounted: {DRIVE_PROJECT_ROOT.parent}")


## 1.3 System packages


In [ ]:
import shutil

if shutil.which("zstd") is None:
    !sudo apt-get update
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError("Failed to update apt package index.")
    !sudo apt-get install -y zstd
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError("Failed to install zstd.")
else:
    print("zstd is already available.")

!zstd --version
if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("zstd command is not available after preflight.")


## 1.4 Repository checkout


In [ ]:
%cd /content

if PROJECT_ROOT.exists():
    print(f"Removing stale repository checkout: {PROJECT_ROOT}")
    !rm -rf {PROJECT_ROOT}
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError(f"Failed to remove existing directory: {PROJECT_ROOT}")

!git clone {REPO_URL} {PROJECT_ROOT}
if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("Failed to clone repository.")

!git -C {PROJECT_ROOT} checkout {REPO_REF}
if globals().get("_exit_code", 1) != 0:
    raise RuntimeError(f"Failed to checkout revision {REPO_REF}.")

print(f"Repository ready: {PROJECT_ROOT}")
!git -C {PROJECT_ROOT} rev-parse HEAD
if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("Failed to determine checked out revision.")


## 1.5 Install dependencies


In [ ]:
%cd {PROJECT_ROOT}
%pip install --upgrade pip
%pip install -r "requirements-colab.txt"
print("Repository dependencies installed from requirements-colab.txt.")


## 1.6 Add source tree to import path


In [ ]:
import sys

%cd {PROJECT_ROOT}
source_path = PROJECT_ROOT / "src"
if str(source_path) not in sys.path:
    sys.path.append(str(source_path))
print(f"Repository src directory available on sys.path: {source_path}")


## 1.7 Workflow API imports


In [ ]:
from text_to_sign_production.workflows.foundation.review import display_review_sections
from text_to_sign_production.workflows.debug import (
    DebugSampleRequest,
    DebugWorkflowConfig,
    SampleDebugWorkflow,
)


## 1.8 Build debug workflow config


In [ ]:
debug_config = DebugWorkflowConfig(
    project_root=PROJECT_ROOT,
    drive_project_root=DRIVE_PROJECT_ROOT,
    runtime_root=RUNTIME_ROOT,
)


## 1.9 Instantiate workflow


In [ ]:
workflow = SampleDebugWorkflow(debug_config)
print(f"Debug workflow run id: {workflow.run_id}")


## 1.10 Runtime summary


In [ ]:
import platform

print("Runtime summary:")
print(f"  repo root: {PROJECT_ROOT}")
print(f"  source root: {source_path}")
print(f"  drive project root: {DRIVE_PROJECT_ROOT}")
print(f"  runtime root: {RUNTIME_ROOT}")
print(f"  requested ref: {REPO_REF}")
print(f"  Python: {platform.python_version()}")
print("  imports succeeded: yes")
!git -C {PROJECT_ROOT} rev-parse --abbrev-ref HEAD
!git -C {PROJECT_ROOT} rev-parse --short HEAD


# 2. Operator Inputs


## 2.1 Debug sample request


In [ ]:
DEBUG_SPLITS = ("val",)
TARGET_SENTENCE_NAME = "..."

request = DebugSampleRequest(
    debug_splits=DEBUG_SPLITS,
    target_sentence_name=TARGET_SENTENCE_NAME,
)


## 2.2 Review request


In [ ]:
display_review_sections(workflow.review_request(request))


# 3. Restore Runtime Artifacts


## 3.1 Build restore plan


In [ ]:
restore_plan = workflow.build_restore_plan(request.debug_splits)


## 3.2 Review restore plan


In [ ]:
display_review_sections(workflow.review_restore_plan(restore_plan))


## 3.3 Validate restore plan


In [ ]:
restore_plan_validation = workflow.validate_restore_plan(restore_plan)


## 3.4 Review restore plan validation


In [ ]:
display_review_sections(workflow.review_restore_plan_validation(restore_plan_validation))


## 3.5 Execute restore


In [ ]:
if not restore_plan_validation.valid:
    raise RuntimeError("Restore plan validation failed; stopping before execution.")
restore_result = workflow.execute_restore(restore_plan)


## 3.6 Review restore result


In [ ]:
display_review_sections(workflow.review_restore_result(restore_result))


## 3.7 Verify restored runtime


In [ ]:
runtime_verification = workflow.verify_restored_runtime(restore_plan)


## 3.8 Review runtime verification


In [ ]:
display_review_sections(workflow.review_runtime_verification(runtime_verification))


## 3.9 Hard-stop guard


In [ ]:
if not restore_result.succeeded:
    raise RuntimeError("Required runtime restore failed; stopping debug notebook.")
if not runtime_verification.succeeded:
    raise RuntimeError("Required runtime verification failed; stopping debug notebook.")


# 4. Resolve Target Sample


## 4.1 Resolve target sample


In [ ]:
resolution = workflow.resolve_target_sample(request)


## 4.2 Review target resolution


In [ ]:
display_review_sections(workflow.review_target_resolution(resolution))


## 4.3 Hard-stop guard


In [ ]:
if resolution.status.value != "found":
    raise RuntimeError(f"Target resolution did not find exactly one sample: {resolution.status.value}")


# 5. Sample Evidence Dossier


## 5.1 Collect dossier


In [ ]:
dossier = workflow.collect_sample_evidence(resolution)


## 5.2 Review dossier


In [ ]:
display_review_sections(workflow.review_sample_dossier(dossier))


# 6. Gate Debug


## 6.1 Run gate debug


In [ ]:
gate_result = workflow.debug_gate(dossier)


## 6.2 Review gate debug


In [ ]:
display_review_sections(workflow.review_gate_debug(gate_result))


# 7. Tier Debug


## 7.1 Run tier debug


In [ ]:
tier_result = workflow.debug_tier(dossier, gate_result)


## 7.2 Review tier debug


In [ ]:
display_review_sections(workflow.review_tier_debug(tier_result))


# 8. Visualization Debug


## 8.1 Run visualization debug


In [ ]:
visual_result = workflow.debug_visualization(dossier, gate_result, tier_result)


## 8.2 Review visualization debug


In [ ]:
display_review_sections(workflow.review_visualization_debug(visual_result))


## 8.3 Display visual artifacts


In [ ]:
for artifact in visual_result.artifacts:
    if artifact.created:
        display({"text/plain": str(artifact.path)}, raw=True)


# 9. Reports


## 9.1 Write reports


In [ ]:
report_result = workflow.write_debug_reports(
    dossier=dossier,
    gate_result=gate_result,
    tier_result=tier_result,
    visual_result=visual_result,
)


## 9.2 Review report outputs


In [ ]:
display_review_sections(workflow.review_report_outputs(report_result))


# 10. Publish Debug Outputs


## 10.1 Build publish plan


In [ ]:
publish_plan = workflow.build_publish_plan(
    report_result=report_result,
    visual_result=visual_result,
)


## 10.2 Review publish plan


In [ ]:
display_review_sections(workflow.review_publish_plan(publish_plan))


## 10.3 Execute publish


In [ ]:
publish_result = workflow.publish_debug_outputs(publish_plan)


## 10.4 Review publish result


In [ ]:
display_review_sections(workflow.review_publish_result(publish_result))


## 10.5 Verify published outputs


In [ ]:
publish_verification = workflow.verify_published_outputs(publish_result)


## 10.6 Review publish verification


In [ ]:
display_review_sections(workflow.review_publish_verification(publish_verification))


# 11. Final Print


## 11.1 Build final result


In [ ]:
final_result = workflow.build_final_result(
    dossier=dossier,
    gate_result=gate_result,
    tier_result=tier_result,
    visual_result=visual_result,
    report_result=report_result,
    publish_result=publish_result,
    publish_verification=publish_verification,
)


## 11.2 Print final result


In [ ]:
workflow.print_final_result(final_result)
